# FraudaZero — 01. Análise exploratória (EDA)

**Base:** `data/processed/train.csv`, o *bootstrap dataset* (primeiros 80% das transações por `Time`), gerado por `python -m src.train prepare`.
O `stream.csv` (últimos 20%) **não** é analisado aqui: ele é o teste final de generalização e o fluxo do simulador.

**Variáveis:** `Time` (segundos desde a primeira transação), `V1`–`V28` (componentes de PCA já aplicados pelo autor da base, por sigilo), `Amount` (valor) e `Class` (1 = fraude).

As figuras são salvas em `reports/figs/` para o relatório. Cada gráfico tem a interpretação logo abaixo.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
FIGS = ROOT / "reports" / "figs"
FIGS.mkdir(parents=True, exist_ok=True)

# Paleta: slots 1 e 2 da paleta categórica validada (CVD-safe). Legítima = azul, fraude = laranja.
C_LEGIT, C_FRAUD = "#2a78d6", "#eb6834"
plt.rcParams.update(
    {
        "figure.dpi": 110,
        "savefig.dpi": 150,
        "savefig.bbox": "tight",
        "figure.facecolor": "#fcfcfb",
        "axes.facecolor": "#fcfcfb",
        "axes.edgecolor": "#c3c2b7",
        "axes.labelcolor": "#52514e",
        "axes.titlecolor": "#0b0b0b",
        "xtick.color": "#52514e",
        "ytick.color": "#52514e",
        "text.color": "#0b0b0b",
        "axes.grid": True,
        "grid.color": "#e8e7e3",
        "grid.linewidth": 0.8,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "font.size": 10,
    }
)


def save(fig, name):
    fig.savefig(FIGS / f"{name}.png")
    plt.show()


df = pd.read_csv(ROOT / "data" / "processed" / "train.csv")
V = [f"V{i}" for i in range(1, 29)]
legit, fraud = df[df.Class == 0], df[df.Class == 1]
print(df.shape)

## 1. Visão geral: tipos, nulos, duplicados e estatísticas descritivas

In [ ]:
print("Tipos:", df.dtypes.value_counts().to_dict())
print("Nulos no total:", int(df.isna().sum().sum()))
print(
    "Linhas duplicadas:",
    int(df.duplicated().sum()),
    "| entre elas, fraudes:",
    int(df[df.duplicated()].Class.sum()),
)
print(
    "Classes:",
    df.Class.value_counts().to_dict(),
    f"| prevalência de fraude: {df.Class.mean():.4%}",
)
df[["Time", "Amount", "V1", "V2", "V3", "V14", "V17"]].describe().T.round(2)

**Leitura.** São 227.845 transações e 31 colunas numéricas, **sem nenhum valor ausente**. Há **840 cópias exatas** de linhas (18 delas são fraudes), tratadas na seção 4.
`V1`–`V28` têm média perto de 0 (efeito do PCA), mas desvios diferentes entre si (de 0,33 a 1,94), e caudas longas (ex.: `V2` vai até 22). `Amount` tem média 90,82 e mediana 23,40: a distribuição é muito assimétrica, com máximo de 19.656,53.

In [ ]:
stats = df.groupby("Class")["Amount"].describe(percentiles=[0.5, 0.9, 0.99]).round(2)
stats.index = ["legítima", "fraude"]
print(
    "Assimetria (skew) de Amount:",
    round(df.Amount.skew(), 2),
    "| após log1p:",
    round(np.log1p(df.Amount).skew(), 2),
)
print(
    "Transações com Amount = 0:",
    int((df.Amount == 0).sum()),
    "| fraudes com Amount = 0:",
    int((fraud.Amount == 0).sum()),
)
stats

**Leitura.** A assimetria de `Amount` é 14,07; depois de `log1p` cai para 0,14, praticamente simétrica. Fraudes têm **mediana menor** (12,31 contra 23,42) e **máximo menor** (2.125,87 contra 19.656,53), mas média maior (125,66 contra 90,76): concentram-se em valores muito baixos e em uma faixa intermediária. 1.567 transações têm valor 0, das quais 22 são fraudes.

## 2. Visualizações

In [ ]:
counts = df.Class.value_counts().sort_index()
fig, ax = plt.subplots(figsize=(6, 3.2))
bars = ax.bar(
    ["Legítima (0)", "Fraude (1)"], counts.values, color=[C_LEGIT, C_FRAUD], width=0.5
)
ax.set_yscale("log")
ax.set_ylabel("Transações (escala log)")
ax.set_title("Fig. 1 — Distribuição das classes no treino")
for b, n in zip(bars, counts.values):
    ax.annotate(
        f"{n:,}".replace(",", ".") + f"  ({n / len(df):.2%})",
        (b.get_x() + b.get_width() / 2, n),
        ha="center",
        va="bottom",
        fontsize=9,
    )
ax.grid(axis="x", visible=False)
save(fig, "eda_01_classes")

**Fig. 1 — Interpretação.** O desbalanceamento é extremo: **417 fraudes (0,18%) contra 227.428 legítimas**, cerca de 1 fraude a cada 546 transações (o eixo está em escala log; em escala linear a barra de fraude seria invisível). Consequências diretas:
- **Acurácia não serve como métrica**: prever "legítima" para tudo dá 99,82% de acerto e zero fraudes detectadas. Usamos **PR-AUC** como principal, mais recall, precisão e F1 da classe fraude.
- A divisão em folds precisa ser **estratificada**, senão alguns folds ficam com pouquíssimas fraudes.
- O modelo precisa de tratamento do desbalanceamento (`class_weight` ou SMOTE), sempre **dentro do Pipeline**, só nos dados de treino.

In [ ]:
bins = np.linspace(0, np.log10(df.Amount.max() + 1), 50)
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.hist(
    np.log10(legit.Amount + 1),
    bins=bins,
    density=True,
    histtype="step",
    lw=2,
    color=C_LEGIT,
    label=f"Legítima (mediana {legit.Amount.median():.2f})",
)
ax.hist(
    np.log10(fraud.Amount + 1),
    bins=bins,
    density=True,
    histtype="step",
    lw=2,
    color=C_FRAUD,
    label=f"Fraude (mediana {fraud.Amount.median():.2f})",
)
ax.set_xlabel("log10(Amount + 1)")
ax.set_ylabel("Densidade")
ax.set_title("Fig. 2 — Valor da transação por classe (escala log)")
ax.legend(frameon=False)
save(fig, "eda_02_amount")

**Fig. 2 — Interpretação.** O valor sozinho não separa as classes (as curvas se sobrepõem em toda a faixa), mas as formas são diferentes:
- **Pico de fraudes em valores mínimos:** 157 das 417 fraudes (37,6%) têm valor até 1,50, sendo 105 de exatamente 1,00. Entre as legítimas, só 12,2% estão nessa faixa. O padrão é compatível com o "teste de cartão" (compra pequena para checar se o cartão roubado funciona).
- **Segundo pico perto de 100:** 44 fraudes entre 90 e 110, sendo 27 de exatamente 99,99.
- As fraudes não chegam aos valores mais altos (máximo de 2.125,87).

Como a relação é **não linear** (picos, e não "quanto maior, mais fraude"), `Amount` quase não tem correlação linear com a classe (Fig. 5), mas deve continuar no modelo. A escala log é a adequada: na escala linear, 99% dos pontos ficam espremidos à esquerda.

In [ ]:
hours = (df.Time // 3600).astype(int)
vol = hours.value_counts().sort_index()
fr = df.groupby(hours).Class.sum()
rate = df.groupby(hours).Class.mean() * 100

fig, axes = plt.subplots(2, 1, figsize=(8, 5), sharex=True)
axes[0].bar(vol.index, vol.values, color=C_LEGIT, width=0.85)
axes[0].set_ylabel("Transações / hora")
axes[0].set_title(
    "Fig. 3 — Volume e taxa de fraude ao longo de Time (horas desde o início)"
)
axes[1].bar(rate.index, rate.values, color=C_FRAUD, width=0.85)
axes[1].set_ylabel("% de fraude na hora")
axes[1].set_xlabel("Hora desde a primeira transação (0–40)")
for ax in axes:
    ax.grid(axis="x", visible=False)
save(fig, "eda_03_tempo")
print("Hora com mais fraudes:", int(fr.idxmax()), f"({int(fr.max())} fraudes)")
print(
    "Maior taxa de fraude:",
    f"{rate.max():.2f}% na hora {int(rate.idxmax())}",
    f"(volume {int(vol[rate.idxmax()])})",
)

**Fig. 3 — Interpretação.** O painel de cima mostra **dois ciclos diários**: o volume cai para 1–2 mil transações/hora nas horas 1–7 e 25–31 e sobe para 8–10 mil nas demais. O painel de baixo mostra que a **taxa de fraude dispara justamente nos vales de volume**: 1,33% na hora 2, 2,05% na hora 26 e 1,51% na hora 28, contra a média de 0,18%. Em números absolutos, a hora com mais fraudes é a 11 (43), que tem volume alto.

Implicação para o modelo: `Time` bruto é só a posição numa janela de 48 horas. Os dados de produção (o stream) têm `Time` entre 145.248 e 172.792 s, valores **que nunca aparecem no treino** (0 a 145.247). Um modelo que use `Time` bruto estaria extrapolando. O que se repete de um dia para o outro é a **hora do dia** (Fig. 4).

In [ ]:
by_hod = df.assign(hod=((df.Time % 86400) // 3600).astype(int)).groupby("hod")
rate_hod = by_hod.Class.mean() * 100
fig, ax = plt.subplots(figsize=(8, 3.2))
ax.bar(rate_hod.index, rate_hod.values, color=C_FRAUD, width=0.8)
ax.axhline(df.Class.mean() * 100, color="#52514e", lw=1, ls="--")
ax.annotate(
    "média geral",
    (12.6, df.Class.mean() * 100),
    ha="left",
    va="bottom",
    fontsize=9,
    color="#52514e",
)
ax.set_xticks(range(0, 24, 2))
ax.set_xlabel("Hora do dia (Time % 86400, em horas)")
ax.set_ylabel("% de fraude")
ax.set_title("Fig. 4 — Taxa de fraude por hora do dia (os dois dias somados)")
ax.grid(axis="x", visible=False)
save(fig, "eda_04_hora_do_dia")
print("Taxa por hora do dia (%):", rate_hod.round(3).to_dict())

**Fig. 4 — Interpretação.** Somando os dois dias por hora do dia (`Time % 86400`), a madrugada concentra o risco: **1,71% de fraude às 2h** (57 de 3.328 transações), **1,04% às 4h** e 0,49% às 3h, contra 0,03%–0,37% nas demais horas (média geral de 0,18%). Isso justifica transformar `Time` em hora do dia dentro do Pipeline.

Cuidados: (1) não sabemos o horário real em que a coleta começou; o ciclo de volume sugere que `Time = 0` fica perto da meia-noite, mas um deslocamento constante não muda a informação disponível ao modelo; (2) são só dois dias e poucas transações por hora na madrugada, então as taxas noturnas são ruidosas. Por isso a hora entra como **seno e cosseno** (23h fica perto de 0h) e o modelo decide quanto peso dar a ela.

In [ ]:
corr = df[["Time", "Amount"] + V].corrwith(df.Class).sort_values()
fig, ax = plt.subplots(figsize=(7, 7))
ax.barh(
    corr.index,
    corr.values,
    color=[C_FRAUD if v > 0 else C_LEGIT for v in corr.values],
    height=0.7,
)
ax.axvline(0, color="#52514e", lw=0.8)
ax.set_xlabel("Correlação de Pearson com Class")
ax.set_title("Fig. 5 — Correlação de cada variável com a classe fraude")
ax.grid(axis="y", visible=False)
save(fig, "eda_05_correlacao")
print("Mais negativas:", corr.head(5).round(3).to_dict())
print("Mais positivas:", corr.tail(5).round(3).to_dict())

**Fig. 5 — Interpretação.** As variáveis com relação linear mais forte com a fraude são **negativas**: `V17` (−0,364), `V14` (−0,313), `V12` (−0,269), `V10` (−0,240) e `V16` (−0,220). Do lado positivo, `V11` (0,165), `V4` (0,140) e `V2` (0,103). `Amount` e `Time` ficam perto de zero.

Os valores parecem baixos, mas com 0,18% de positivos a correlação de Pearson com uma variável binária tem teto baixo; o que importa é a ordem. Correlação perto de zero **não** significa variável inútil: a Fig. 2 mostra que `Amount` tem relação não linear forte. Por isso **não removemos variáveis** com base nesta figura; modelos não lineares (árvores, RF, AdaBoost, MLP) podem aproveitá-las.

In [ ]:
top = corr.abs().sort_values(ascending=False).index[:4].tolist()
fig, axes = plt.subplots(1, 4, figsize=(12, 3))
for ax, col in zip(axes, top):
    lo, hi = df[col].quantile([0.001, 0.999])
    b = np.linspace(min(lo, fraud[col].min()), hi, 50)
    ax.hist(
        legit[col],
        bins=b,
        density=True,
        histtype="step",
        lw=2,
        color=C_LEGIT,
        label="Legítima",
    )
    ax.hist(
        fraud[col],
        bins=b,
        density=True,
        histtype="step",
        lw=2,
        color=C_FRAUD,
        label="Fraude",
    )
    ax.set_title(col)
    ax.set_yticks([])
axes[0].legend(frameon=False, fontsize=8)
fig.suptitle(
    "Fig. 6 — Distribuição das 4 variáveis V mais correlacionadas com a fraude", y=1.04
)
save(fig, "eda_06_top_v")
print("Top 4:", top)

**Fig. 6 — Interpretação.** Nas quatro variáveis mais correlacionadas, as legítimas formam um pico estreito em torno de 0, enquanto as fraudes se espalham por uma **cauda longa de valores negativos** (até −20 ou menos). Um corte simples, como "`V14` abaixo de −3", já isola boa parte das fraudes, o que favorece árvores de decisão.

Há também uma parcela de fraudes **dentro do pico das legítimas** (perto de 0). Essas são as difíceis: nenhuma das variáveis sozinha as distingue, o que limita o recall alcançável e mostra que o modelo precisa combinar variáveis. As escalas são muito diferentes entre as variáveis, então KNN, MLP e regressão logística precisam de padronização.

In [ ]:
rng = np.random.RandomState(42)
sample = pd.concat([fraud, legit.sample(5000, random_state=rng)])
X2 = StandardScaler().fit_transform(sample[V + ["Amount"]])
pca = PCA(n_components=2, random_state=42).fit(X2)
P = pca.transform(X2)
is_f = sample.Class.values == 1

fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(
    P[~is_f, 0],
    P[~is_f, 1],
    s=8,
    alpha=0.35,
    color=C_LEGIT,
    label="Legítima (5.000, amostra)",
    edgecolors="none",
)
ax.scatter(
    P[is_f, 0],
    P[is_f, 1],
    s=14,
    alpha=0.8,
    color=C_FRAUD,
    label=f"Fraude (todas, {is_f.sum()})",
    edgecolors="#fcfcfb",
    linewidths=0.5,
)
ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.1%} da variância)")
ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.1%} da variância)")
ax.set_title("Fig. 7 — Projeção PCA 2D (V1–V28 + Amount padronizados)")
ax.legend(frameon=False)
save(fig, "eda_07_pca")

**Fig. 7 — Interpretação.** Projetando `V1`–`V28` + `Amount` padronizados em 2 componentes principais (38,2% da variância: 31,4% + 6,8%), as legítimas formam um **núcleo compacto** perto da origem, e a maior parte das fraudes se afasta dele ao longo de PC1, formando estruturas alongadas. Isso sugere que muitas fraudes são separáveis (ROC-AUC alta é esperada).

Porém, **parte das fraudes cai dentro do núcleo das legítimas**, coerente com a Fig. 6. Essa sobreposição é o que deve impedir a PR-AUC de chegar perto de 1. Como a amostra tem todas as 417 fraudes e só 5.000 legítimas, a figura exagera a presença de fraudes (7,7% da amostra contra 0,18% na base): serve para ver a forma, não a proporção.

## 3. Duplicados

In [ ]:
dups = df[df.duplicated(keep=False)]
print(
    "Linhas envolvidas em duplicação:",
    len(dups),
    "| cópias extras (removíveis):",
    int(df.duplicated().sum()),
)
print(
    "Fraudes antes:",
    int(df.Class.sum()),
    "| depois de remover cópias:",
    int(df.drop_duplicates().Class.sum()),
)
print(
    "Rótulos conflitantes (mesmas features, classes diferentes):",
    int(
        df.drop_duplicates()
        .duplicated(subset=[c for c in df.columns if c != "Class"])
        .sum()
    ),
)

**Leitura.** 1.462 linhas participam de alguma duplicação, o que gera 840 cópias extras. Removê-las deixa **227.005 linhas e 399 fraudes**. Nenhum par de linhas idênticas tem rótulos diferentes, então a remoção não cria ambiguidade.

## 4. Tratamento dos dados (decisões e justificativas)

Toda transformação que **aprende algo dos dados** (média, desvio, reamostragem) fica **dentro do `Pipeline`** (`src/features.py` e `src/train.py`) e, portanto, é ajustada só nos dados de treino de cada fold. Ajustar a escala ou reamostrar antes do split vazaria informação da validação para o treino.

| # | Tema | Decisão | Justificativa |
|---|---|---|---|
| 1 | **Ausentes** | Nada a fazer | 0 valores nulos (seção 1). A checagem fica registrada. |
| 2 | **Duplicados** | Remover as 840 cópias exatas do **treino** (`validation.load_train`). O stream fica intacto. | No k-fold, uma cópia pode cair no treino e a outra na validação: o modelo "acerta" por memorização e a métrica fica otimista (afeta principalmente KNN e árvores). As cópias não trazem informação nova e não há rótulos conflitantes. A remoção fica **fora** do Pipeline porque não dá para "remover linhas" na inferência, e não há vazamento: é uma regra fixa, sem estatística aprendida. |
| 3 | **`Amount`: escala** | `log1p` seguido de `StandardScaler`, dentro do Pipeline | Assimetria 14,07 → 0,14 depois de `log1p`. Modelos por distância (KNN), gradiente (MLP) e lineares são sensíveis à escala; para árvores é indiferente. |
| 4 | **`Amount`: outliers** | **Não remover** | Os valores extremos são legítimos (não são erro de digitação), e as fraudes ocupam faixas específicas (Fig. 2). Remover outliers removeria sinal. O `log1p` já reduz a influência dos extremos nos modelos sensíveis à escala. |
| 5 | **`Time`** | Converter em **hora do dia** (`Time % 86400`) codificada em **seno e cosseno**, dentro do Pipeline | `Time` bruto não se repete: o stream tem valores (145.248–172.792 s) que o treino nunca viu (Fig. 3). A hora do dia se repete e carrega sinal (Fig. 4). Seno e cosseno deixam 23h perto de 0h. **O contrato não muda:** a API continua recebendo `Time` bruto; a transformação acontece dentro do modelo. |
| 6 | **`V1`–`V28`** | `StandardScaler`, dentro do Pipeline | Já são componentes de PCA (média ~0), mas os desvios variam de 0,33 a 1,94. Padronizar dá o mesmo peso a todas no KNN/MLP. Nenhuma variável é removida (Fig. 5). |
| 7 | **Desbalanceamento** | `class_weight="balanced"` no baseline; SMOTE ou undersampling **só** via `imblearn.pipeline.Pipeline` | Garante que a reamostragem acontece só nos dados de treino de cada fold. A validação mantém a proporção real de fraudes, que é a que importa para a PR-AUC. |

A célula abaixo mostra o pré-processamento aplicado a algumas linhas (ajustado aqui só para ilustrar).

In [ ]:
import sys

sys.path.insert(0, str(ROOT))
from src.features import preprocessor

prep = preprocessor().fit(df.drop(columns="Class"))
pd.DataFrame(
    prep.transform(df.drop(columns="Class").head(5)),
    columns=prep.get_feature_names_out(),
).round(3).iloc[:, :6]

## 5. Resumo para o relatório

- 227.845 transações no treino, **0,18% de fraudes**, sem ausentes; 840 cópias exatas removidas (restam 227.005 e 399 fraudes).
- Fraudes se concentram em **valores muito baixos** (37,6% até 1,50) e na **madrugada** (1,71% às 2h contra 0,18% em média).
- As variáveis mais discriminantes são `V17`, `V14`, `V12` e `V10`, com fraudes na cauda negativa. Parte das fraudes se sobrepõe às legítimas em todas as vistas: é o limite do problema.
- Tratamento: hora do dia cíclica, `log1p(Amount)`, padronização e desbalanceamento, tudo **dentro do Pipeline**; nenhum outlier ou variável removido.